# t-SNE Analysis of TerraMind Embeddings
## Comparing Real vs Simulated Data Fine-tuned Models

This notebook loads two fine-tuned TerraMind models (trained on real and simulated data) and visualizes their learned embeddings using t-SNE dimensionality reduction.

## 1. Import Required Libraries

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import logging
from typing import Tuple, Dict, List
import pandas as pd
import seaborn as sns
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

: 

## 2. Load Pre-trained Models

Load the two fine-tuned SemanticSegmentationTask models trained on real and simulated data.

In [5]:
from lightning.pytorch import Trainer
from terratorch.tasks import SemanticSegmentationTask
from terratorch.datamodules.sen1floods11 import Sen1Floods11NonGeoDataModule
from terra_sat_drift.data_simulation.phisat2_constants import S2_BANDS_NAMES, S2_BANDS, S2_PAN_BANDS

# Configuration
config = {
    'backbone_size': 'tiny',
    'num_classes': 2,
    'class_names': ['background', 'flood'],
    # Update these paths to your actual checkpoint locations
    'real_data_checkpoint': '/shared/home/elucas/terra-sat-drift/outputs/terramind_sen1floods/terramind_v1_tiny/checkpoints/best-val_mIoU.ckpt',
    'simulated_data_checkpoint': '/shared/home/elucas/terra-sat-drift/outputs/terramind_sen1floods_simulated/terramind_v1_tiny_simulated/checkpoints/best-val_mIoU.ckpt',
    'data_root': '/shared/home/elucas/datasets/sen1floods11',
}

def create_model() -> SemanticSegmentationTask:
    """Create a SemanticSegmentationTask model with TerraMind backbone."""
    BACKBONE_NECK_INDICES = {
        "tiny": [1, 3, 4, 5],
        "small": [1, 3, 4, 5],  
        "base": [2, 5, 8, 11],
        "large": [5, 11, 17, 23],
    }

    
    backbone_name = f"terramind_v1_{config['backbone_size']}"
    neck_indices = BACKBONE_NECK_INDICES[config['backbone_size']]

    model_args = {
        "backbone": backbone_name,
        "backbone_pretrained": True,
        "backbone_modalities": ["S2L1C"],
        "backbone_bands": {"S2L1C": ["B02", "B03", "B04", "B08", "B05", "B06", "B07"]},
        "necks": [
            {"name": "SelectIndices", "indices": neck_indices},
            {"name": "ReshapeTokensToImage", "remove_cls_token": False},
            {"name": "LearnedInterpolateToPyramidal"},
        ],
        "decoder": "UNetDecoder",
        "decoder_channels": [256, 128, 64, 32],
        "head_dropout": 0.1,
        "num_classes": config['num_classes'],
    }
    
    return SemanticSegmentationTask(
        model_factory="EncoderDecoderFactory",
        model_args=model_args,
        lr=1e-4,
        ignore_index=-1,
        plot_on_val=False,
        loss="dice",
        optimizer="AdamW",
        class_names=config['class_names'],
        freeze_backbone=False,
    )

# Load models from checkpoints
logger.info("Loading models from checkpoints...")

model_real = create_model()
model_simulated = create_model()

try:
    real_ckpt = torch.load(config['real_data_checkpoint'], map_location=device)
    model_real.load_state_dict(real_ckpt['state_dict'])
    logger.info(f"✓ Loaded real data model from {config['real_data_checkpoint']}")
except Exception as e:
    logger.warning(f"Could not load real data checkpoint: {e}")
    logger.info("Using initialized model instead")

try:
    sim_ckpt = torch.load(config['simulated_data_checkpoint'], map_location=device)
    model_simulated.load_state_dict(sim_ckpt['state_dict'])
    logger.info(f"✓ Loaded simulated data model from {config['simulated_data_checkpoint']}")
except Exception as e:
    logger.warning(f"Could not load simulated data checkpoint: {e}")
    logger.info("Using initialized model instead")

# Move models to device and set to eval mode
model_real = model_real.to(device).eval()
model_simulated = model_simulated.to(device).eval()

logger.info("Models loaded and set to evaluation mode")

2026-05-13 16:12:21,676 - INFO - Loading models from checkpoints...
2026-05-13 16:12:22,224 - INFO - HTTP Request: HEAD https://huggingface.co/ibm-esa-geospatial/TerraMind-1.0-tiny/resolve/main/TerraMind_v1_tiny.pt "HTTP/1.1 302 Found"
2026-05-13 16:12:24,458 - INFO - HTTP Request: HEAD https://huggingface.co/ibm-esa-geospatial/TerraMind-1.0-tiny/resolve/main/TerraMind_v1_tiny.pt "HTTP/1.1 302 Found"
2026-05-13 16:12:27,220 - INFO - ✓ Loaded real data model from /shared/home/elucas/terra-sat-drift/outputs/terramind_sen1floods/terramind_v1_tiny/checkpoints/best-val_mIoU.ckpt
2026-05-13 16:12:29,039 - WARNING - Could not load simulated data checkpoint: Error(s) in loading state_dict for SemanticSegmentationTask:
	size mismatch for model.encoder.encoder_embeddings.untok_sen2l1c@224.proj.weight: copying a param with shape torch.Size([192, 2048]) from checkpoint, the shape in current model is torch.Size([192, 1792]).
2026-05-13 16:12:29,040 - INFO - Using initialized model instead
2026-05-1

## 3. Extract Embeddings from Data

Create a data loader and extract intermediate embeddings from both models.

In [ ]:
class EmbeddingExtractor:
    """Utility class to extract embeddings from models."""
    
    def __init__(self, model: SemanticSegmentationTask, device: torch.device):
        self.model = model
        self.device = device
        self.embeddings = None
        self.labels = None
        
    def extract_embeddings(self, dataloader, max_batches: int = None) -> Tuple[np.ndarray, np.ndarray]:
        """
        Extract embeddings from the encoder before the classification head.
        
        Args:
            dataloader: DataLoader with (images, masks) tuples
            max_batches: Maximum number of batches to process (None for all)
            
        Returns:
            Tuple of (embeddings, labels)
        """
        all_embeddings = []
        all_labels = []
        
        with torch.no_grad():
            for batch_idx, batch in enumerate(tqdm(dataloader, desc="Extracting embeddings")):
                if max_batches and batch_idx >= max_batches:
                    break
                
                # Handle different batch formats
                if isinstance(batch, (tuple, list)):
                    x = batch[0]  # images
                    y = batch[1]  # masks
                elif isinstance(batch, dict):
                    x = batch.get('image') or batch.get('images')
                    y = batch.get('mask') or batch.get('masks') or batch.get('label')
                else:
                    x = batch
                    y = None
                
                x = x.to(self.device)
                if y is not None:
                    y = y.to(self.device)
                
                # Forward pass
                output = self.model(x)
                
                # Extract spatial features (before head)
                # For SemanticSegmentationTask, we need to extract from the encoder
                if hasattr(self.model, 'backbone'):
                    with torch.no_grad():
                        backbone_out = self.model.backbone(x)
                        # Flatten spatial dimensions to get feature vectors
                        if isinstance(backbone_out, torch.Tensor):
                            batch_embeddings = backbone_out.reshape(backbone_out.shape[0], -1)
                        else:
                            # Handle tuple output
                            batch_embeddings = backbone_out[-1].reshape(backbone_out[-1].shape[0], -1)
                else:
                    # Alternative: use model output logits as embeddings
                    batch_embeddings = output.reshape(output.shape[0], -1)
                
                all_embeddings.append(batch_embeddings.cpu().numpy())
                
                if y is not None:
                    # Flatten mask and take mode of each spatial location as class label
                    batch_labels = y.reshape(y.shape[0], -1).mode(dim=1)[0].cpu().numpy()
                    all_labels.append(batch_labels)
        
        embeddings = np.concatenate(all_embeddings, axis=0)
        labels = np.concatenate(all_labels, axis=0) if all_labels else None
        
        return embeddings, labels

# Create data module
logger.info("Creating data module...")
datamodule = Sen1Floods11NonGeoDataModule(
    data_root=config['data_root'],
    bands=S2_BANDS_NAMES,
    num_workers=2,
    batch_size=8,
    download=False,
    use_metadata=True
)
datamodule.setup(stage="validate")

# Extract embeddings from both models
logger.info("Extracting embeddings from real data model...")
extractor_real = EmbeddingExtractor(model_real, device)
embeddings_real, labels_real = extractor_real.extract_embeddings(
    datamodule.val_dataloader(),
    max_batches=50  # Limit for faster processing
)

logger.info(f"Real data embeddings shape: {embeddings_real.shape}")
logger.info(f"Real data labels shape: {labels_real.shape if labels_real is not None else 'N/A'}")

logger.info("Extracting embeddings from simulated data model...")
extractor_sim = EmbeddingExtractor(model_simulated, device)
embeddings_sim, labels_sim = extractor_sim.extract_embeddings(
    datamodule.val_dataloader(),
    max_batches=50  # Limit for faster processing
)

logger.info(f"Simulated data embeddings shape: {embeddings_sim.shape}")
logger.info(f"Simulated data labels shape: {labels_sim.shape if labels_sim is not None else 'N/A'}")

## 4. Apply t-SNE Dimensionality Reduction

Use scikit-learn's TSNE to reduce embeddings to 2D for visualization.

In [ ]:
# Standardize embeddings before t-SNE
logger.info("Standardizing embeddings...")
scaler = StandardScaler()

# Combine embeddings for consistent scaling
all_embeddings = np.vstack([embeddings_real, embeddings_sim])
all_embeddings_scaled = scaler.fit_transform(all_embeddings)

# Split back
embeddings_real_scaled = all_embeddings_scaled[:len(embeddings_real)]
embeddings_sim_scaled = all_embeddings_scaled[len(embeddings_real):]

logger.info("Embeddings standardized")

# Apply t-SNE
logger.info("Applying t-SNE to real data embeddings...")
tsne_real = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate=200,
    n_iter=1000,
    random_state=42,
    verbose=1
)
embeddings_real_tsne = tsne_real.fit_transform(embeddings_real_scaled)
logger.info(f"Real data t-SNE shape: {embeddings_real_tsne.shape}")

logger.info("Applying t-SNE to simulated data embeddings...")
tsne_sim = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate=200,
    n_iter=1000,
    random_state=42,
    verbose=1
)
embeddings_sim_tsne = tsne_sim.fit_transform(embeddings_sim_scaled)
logger.info(f"Simulated data t-SNE shape: {embeddings_sim_tsne.shape}")

## 5. Visualize Embeddings

Create scatter plots of t-SNE reduced embeddings colored by class labels.

In [ ]:
# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Define colors for classes
colors = ['#2E86AB', '#A23B72']  # blue for background, magenta for flood
class_names = ['Background', 'Flood']

# Plot real data embeddings
if labels_real is not None:
    for class_idx, (color, class_name) in enumerate(zip(colors, class_names)):
        mask = labels_real == class_idx
        axes[0].scatter(
            embeddings_real_tsne[mask, 0],
            embeddings_real_tsne[mask, 1],
            c=color,
            label=class_name,
            alpha=0.6,
            s=30,
            edgecolors='black',
            linewidth=0.5
        )
else:
    axes[0].scatter(
        embeddings_real_tsne[:, 0],
        embeddings_real_tsne[:, 1],
        c='#2E86AB',
        alpha=0.6,
        s=30,
        edgecolors='black',
        linewidth=0.5
    )

axes[0].set_title('TerraMind: Real Data Fine-tuned\nt-SNE Embedding Visualization', fontsize=14, fontweight='bold')
axes[0].set_xlabel('t-SNE Component 1', fontsize=12)
axes[0].set_ylabel('t-SNE Component 2', fontsize=12)
axes[0].legend(loc='best', fontsize=11)
axes[0].grid(True, alpha=0.3)

# Plot simulated data embeddings
if labels_sim is not None:
    for class_idx, (color, class_name) in enumerate(zip(colors, class_names)):
        mask = labels_sim == class_idx
        axes[1].scatter(
            embeddings_sim_tsne[mask, 0],
            embeddings_sim_tsne[mask, 1],
            c=color,
            label=class_name,
            alpha=0.6,
            s=30,
            edgecolors='black',
            linewidth=0.5
        )
else:
    axes[1].scatter(
        embeddings_sim_tsne[:, 0],
        embeddings_sim_tsne[:, 1],
        c='#A23B72',
        alpha=0.6,
        s=30,
        edgecolors='black',
        linewidth=0.5
    )

axes[1].set_title('TerraMind: Simulated Data Fine-tuned\nt-SNE Embedding Visualization', fontsize=14, fontweight='bold')
axes[1].set_xlabel('t-SNE Component 1', fontsize=12)
axes[1].set_ylabel('t-SNE Component 2', fontsize=12)
axes[1].legend(loc='best', fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/shared/home/elucas/terra-sat-drift/outputs/tsne_embeddings_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

logger.info("✓ Visualization saved to outputs/tsne_embeddings_comparison.png")

## 6. Compare Real vs Simulated Data Embeddings

Generate comparison visualizations and compute metrics for cluster analysis.

In [ ]:
from scipy.spatial.distance import pdist, squareform
from scipy.stats import entropy

def compute_cluster_metrics(embeddings: np.ndarray, labels: np.ndarray) -> Dict[str, float]:
    """Compute metrics for clustering quality."""
    metrics = {}
    
    if labels is None or len(np.unique(labels)) < 2:
        return metrics
    
    # Compute within-class and between-class distances
    unique_labels = np.unique(labels)
    
    within_class_distances = []
    between_class_distances = []
    
    for label in unique_labels:
        class_embeddings = embeddings[labels == label]
        if len(class_embeddings) > 1:
            # Within-class distances
            distances = pdist(class_embeddings, metric='euclidean')
            within_class_distances.extend(distances)
    
    # Between-class distances
    for i, label1 in enumerate(unique_labels):
        for label2 in unique_labels[i+1:]:
            emb1 = embeddings[labels == label1]
            emb2 = embeddings[labels == label2]
            distances = pdist(np.vstack([emb1, emb2]), metric='euclidean')
            # Take distances between the two classes
            n1, n2 = len(emb1), len(emb2)
            between_distances = distances[n1*n2:]
            between_class_distances.extend(between_distances)
    
    if within_class_distances:
        metrics['mean_within_class_distance'] = np.mean(within_class_distances)
        metrics['std_within_class_distance'] = np.std(within_class_distances)
    
    if between_class_distances:
        metrics['mean_between_class_distance'] = np.mean(between_class_distances)
        metrics['std_between_class_distance'] = np.std(between_class_distances)
    
    # Cluster separation ratio (higher is better)
    if within_class_distances and between_class_distances:
        metrics['separation_ratio'] = (np.mean(between_class_distances) / 
                                      (np.mean(within_class_distances) + 1e-6))
    
    return metrics

# Compute metrics
logger.info("Computing cluster metrics...")

metrics_real = compute_cluster_metrics(embeddings_real_scaled, labels_real)
metrics_sim = compute_cluster_metrics(embeddings_sim_scaled, labels_sim)

# Create comparison table
comparison_data = []
all_metrics = set(list(metrics_real.keys()) + list(metrics_sim.keys()))

for metric in sorted(all_metrics):
    comparison_data.append({
        'Metric': metric,
        'Real Data Model': f"{metrics_real.get(metric, 'N/A'):.4f}" if metric in metrics_real else 'N/A',
        'Simulated Data Model': f"{metrics_sim.get(metric, 'N/A'):.4f}" if metric in metrics_sim else 'N/A'
    })

comparison_df = pd.DataFrame(comparison_data)
print("\n" + "="*80)
print("EMBEDDING QUALITY METRICS COMPARISON")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80 + "\n")

# Visualize metrics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Within-class distances
if 'mean_within_class_distance' in metrics_real and 'mean_within_class_distance' in metrics_sim:
    models = ['Real Data', 'Simulated Data']
    within_distances = [
        metrics_real.get('mean_within_class_distance', 0),
        metrics_sim.get('mean_within_class_distance', 0)
    ]
    colors_bars = ['#2E86AB', '#A23B72']
    axes[0].bar(models, within_distances, color=colors_bars, alpha=0.7, edgecolor='black', linewidth=2)
    axes[0].set_ylabel('Mean Distance', fontsize=12)
    axes[0].set_title('Within-Class Distance\n(Lower is Better)', fontsize=13, fontweight='bold')
    axes[0].grid(True, alpha=0.3, axis='y')

# Cluster separation ratio
if 'separation_ratio' in metrics_real and 'separation_ratio' in metrics_sim:
    separation_ratios = [
        metrics_real.get('separation_ratio', 0),
        metrics_sim.get('separation_ratio', 0)
    ]
    axes[1].bar(models, separation_ratios, color=colors_bars, alpha=0.7, edgecolor='black', linewidth=2)
    axes[1].set_ylabel('Separation Ratio', fontsize=12)
    axes[1].set_title('Cluster Separation Ratio\n(Higher is Better)', fontsize=13, fontweight='bold')
    axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/shared/home/elucas/terra-sat-drift/outputs/embedding_metrics_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

logger.info("✓ Metrics visualization saved to outputs/embedding_metrics_comparison.png")

In [ ]:
# Create combined overlay visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Real data - all points
if labels_real is not None:
    for class_idx, (color, class_name) in enumerate(zip(colors, class_names)):
        mask = labels_real == class_idx
        axes[0, 0].scatter(
            embeddings_real_tsne[mask, 0],
            embeddings_real_tsne[mask, 1],
            c=color,
            label=f'{class_name} (n={np.sum(mask)})',
            alpha=0.6,
            s=40,
            edgecolors='black',
            linewidth=0.5
        )

axes[0, 0].set_title('Real Data Model - Embeddings by Class', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('t-SNE Component 1', fontsize=11)
axes[0, 0].set_ylabel('t-SNE Component 2', fontsize=11)
axes[0, 0].legend(loc='best', fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# Simulated data - all points
if labels_sim is not None:
    for class_idx, (color, class_name) in enumerate(zip(colors, class_names)):
        mask = labels_sim == class_idx
        axes[0, 1].scatter(
            embeddings_sim_tsne[mask, 0],
            embeddings_sim_tsne[mask, 1],
            c=color,
            label=f'{class_name} (n={np.sum(mask)})',
            alpha=0.6,
            s=40,
            edgecolors='black',
            linewidth=0.5
        )

axes[0, 1].set_title('Simulated Data Model - Embeddings by Class', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('t-SNE Component 1', fontsize=11)
axes[0, 1].set_ylabel('t-SNE Component 2', fontsize=11)
axes[0, 1].legend(loc='best', fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

# Distribution statistics for real data
real_data_stats = f"""Real Data Model Statistics:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Total samples: {len(embeddings_real):,}
Embedding dim: {embeddings_real.shape[1]}
t-SNE dim: {embeddings_real_tsne.shape[1]}

Class distribution:
"""
if labels_real is not None:
    for class_idx, class_name in enumerate(class_names):
        count = np.sum(labels_real == class_idx)
        pct = 100 * count / len(labels_real)
        real_data_stats += f"  {class_name}: {count:,} ({pct:.1f}%)\n"

real_data_stats += f"""
Within-class distance: {metrics_real.get('mean_within_class_distance', 'N/A')}
Between-class distance: {metrics_real.get('mean_between_class_distance', 'N/A')}
Separation ratio: {metrics_real.get('separation_ratio', 'N/A'):.4f}
"""

axes[1, 0].text(0.1, 0.5, real_data_stats, fontsize=11, family='monospace',
                verticalalignment='center', transform=axes[1, 0].transAxes,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
axes[1, 0].axis('off')

# Distribution statistics for simulated data
sim_data_stats = f"""Simulated Data Model Statistics:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Total samples: {len(embeddings_sim):,}
Embedding dim: {embeddings_sim.shape[1]}
t-SNE dim: {embeddings_sim_tsne.shape[1]}

Class distribution:
"""
if labels_sim is not None:
    for class_idx, class_name in enumerate(class_names):
        count = np.sum(labels_sim == class_idx)
        pct = 100 * count / len(labels_sim)
        sim_data_stats += f"  {class_name}: {count:,} ({pct:.1f}%)\n"

sim_data_stats += f"""
Within-class distance: {metrics_sim.get('mean_within_class_distance', 'N/A')}
Between-class distance: {metrics_sim.get('mean_between_class_distance', 'N/A')}
Separation ratio: {metrics_sim.get('separation_ratio', 'N/A'):.4f}
"""

axes[1, 1].text(0.1, 0.5, sim_data_stats, fontsize=11, family='monospace',
                verticalalignment='center', transform=axes[1, 1].transAxes,
                bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))
axes[1, 1].axis('off')

plt.suptitle('TerraMind Model Comparison: Real vs Simulated Data Fine-tuning', 
             fontsize=15, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('/shared/home/elucas/terra-sat-drift/outputs/tsne_detailed_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

logger.info("✓ Detailed analysis saved to outputs/tsne_detailed_analysis.png")

# Print summary
print("\n" + "="*80)
print("EMBEDDING ANALYSIS SUMMARY")
print("="*80)
print(f"\n✓ Successfully extracted and analyzed embeddings from both models")
print(f"✓ Real data model: {embeddings_real.shape[0]} samples processed")
print(f"✓ Simulated data model: {embeddings_sim.shape[0]} samples processed")
print(f"✓ Applied t-SNE dimensionality reduction (>1000 dimensions → 2D)")
print(f"✓ Generated visualizations and computed clustering metrics")
print(f"\nOutput files saved to: /shared/home/elucas/terra-sat-drift/outputs/")
print("  - tsne_embeddings_comparison.png")
print("  - embedding_metrics_comparison.png")
print("  - tsne_detailed_analysis.png")
print("="*80 + "\n")